# ***Data Gathering Using TMDB API***

In [5]:
import os
from google.colab import userdata
os.environ['API']=userdata.get('API')

In [6]:
import requests
import os
url="https://api.themoviedb.org/3/trending/movie/week"
data=[]
for page in range(1,501):
  params={'api_key':os.environ.get('API'),page:page}
  response=requests.get(url,params=params)
  movies=response.json()['results']
  for movie in movies:
    element={}
    element['id']=movie['id']
    element['title']=movie['title']
    element['overview']=movie['overview']
    element['genre_ids']=movie['genre_ids']
    element['poster_path']=movie['poster_path']
    data.append(element)

In [7]:
import pandas as pd
df=pd.DataFrame(data)
df.head()

,id,title,overview,genre_ids,poster_path
0,1423191,Resident Evil,Medical courier Bryan unwittingly finds himsel...,"[27, 878, 12]",/gAgWJhUSQWCgtKR8170nMkYlBda.jpg
1,1375441,Primetime,"In 2006, To Catch a Predator host Chris Hansen...","[53, 18]",/lmrulvLbmaejTix1YaMxo1oGhH1.jpg
2,1283515,Verity,"Lowen Ashleigh, hired to ghostwrite for best-s...","[9648, 53]",/rmaNMA1seHqpjtnYKzyynFAr1g8.jpg
3,1507696,Apocalypse Z: New World,The Canary Islands have become Europe’s last r...,"[28, 53, 27]",/wXln4DUrwcyjcFPwgQxFPYTlG1Z.jpg
4,1294189,The Mongoose,A falsely accused war hero with nothing to los...,"[28, 53]",/eSS5mvSG84UUuvtbHel5Yu3Wik4.jpg


In [8]:
genre_url="https://api.themoviedb.org/3/genre/movie/list"
PARAMS={'api_key':os.environ.get('API')}
response2=requests.get(genre_url,params=PARAMS)
genres=response2.json()['genres']
genres

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 16, 'name': 'Animation'},
 {'id': 35, 'name': 'Comedy'},
 {'id': 80, 'name': 'Crime'},
 {'id': 99, 'name': 'Documentary'},
 {'id': 18, 'name': 'Drama'},
 {'id': 10751, 'name': 'Family'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 36, 'name': 'History'},
 {'id': 27, 'name': 'Horror'},
 {'id': 10402, 'name': 'Music'},
 {'id': 9648, 'name': 'Mystery'},
 {'id': 10749, 'name': 'Romance'},
 {'id': 878, 'name': 'Science Fiction'},
 {'id': 10770, 'name': 'TV Movie'},
 {'id': 53, 'name': 'Thriller'},
 {'id': 10752, 'name': 'War'},
 {'id': 37, 'name': 'Western'}]

In [9]:
genres[14]['name'].lower().replace(' ','_')

'science_fiction'

In [10]:
def replace_genres(value):
  L=[]
  for genre in value:
    for i in genres:
      if genre==i['id']:
        L.append(i['name'].lower().replace(' ','_'))
  return " ".join(L)
df['genres']=df['genre_ids'].apply(replace_genres)

In [11]:
df['genres'][0]

'horror science_fiction adventure'

In [12]:
df.drop(columns=['genre_ids'],inplace=True)

In [13]:
df.sample()

,id,title,overview,poster_path,genres
1346,1400940,Clayface,Matt Hagen's meteoric rise from Gotham street ...,/9l1NuFjel2j89hiuiJbsxfblqjy.jpg,horror thriller science_fiction


In [14]:
df.to_csv('movies.csv')

# Data Preprocessing

In [15]:
df.sample(3)

,id,title,overview,poster_path,genres
1534,1727734,Ali G: Who Iz I?,"After twenty years spent ""chillin', tokin' and...",/uTyVxPeWX8Ai30IQoyDDQONNcq.jpg,comedy
8534,1727734,Ali G: Who Iz I?,"After twenty years spent ""chillin', tokin' and...",/uTyVxPeWX8Ai30IQoyDDQONNcq.jpg,comedy
3484,1294189,The Mongoose,A falsely accused war hero with nothing to los...,/eSS5mvSG84UUuvtbHel5Yu3Wik4.jpg,action thriller


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           10000 non-null  int64 
 1   title        10000 non-null  object
 2   overview     10000 non-null  object
 3   poster_path  10000 non-null  object
 4   genres       10000 non-null  object
dtypes: int64(1), object(4)
memory usage: 390.8+ KB


In [17]:
df.isnull().sum()

,0
id,0
title,0
overview,0
poster_path,0
genres,0


In [18]:
df=df[df['poster_path'].isnull()!=True]

In [19]:
df.duplicated().sum()

np.int64(9980)

In [20]:
df.drop_duplicates(inplace=True)

In [21]:
df.shape

(20, 5)

In [22]:
df['overview'][0]

'Medical courier Bryan unwittingly finds himself fighting for survival as one fateful, horrifying night collapses around him in chaos.'

In [23]:
import re
def remove_impurities(col):
  col=col.lower()
  col=re.sub('<.*?>',' ',col)
  col=re.sub('http?://\S+|www\.\S+',' ',col)
  return re.sub('[^a-zA-Z]',' ',col)
df['overview']=df['overview'].apply(remove_impurities)

<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:5: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_1124/3610214272.py:5: SyntaxWarning: invalid escape sequence '\S'
  col=re.sub('http?://\S+|www\.\S+',' ',col)


In [24]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
words=stopwords.words('english')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [25]:
def extraction(col):
  ps=PorterStemmer()
  tokens=word_tokenize(col)
  L=[]
  for i in tokens:
    if i not in words:
      L.append(ps.stem(i))
  return L
df['overview']=df['overview'].apply(extraction)

In [26]:
print(df['overview'][0])

['medic', 'courier', 'bryan', 'unwittingli', 'find', 'fight', 'surviv', 'one', 'fate', 'horrifi', 'night', 'collaps', 'around', 'chao']


In [27]:
df['genres']=df['genres'].apply(lambda x: x.split())

In [28]:
df.sample()

,id,title,overview,poster_path,genres
11,1465063,Forgotten Island,"[celebr, last, night, togeth, high, school, gr...",/myd1K3gefdNAWFsGNRDdMeiK9Qg.jpg,"[animation, adventure, fantasy, comedy, family]"


In [29]:
df['tags']=df['overview']+df['genres']
df.drop(columns=['overview','genres'],inplace=True)

In [30]:
df.head(3)

,id,title,poster_path,tags
0,1423191,Resident Evil,/gAgWJhUSQWCgtKR8170nMkYlBda.jpg,"[medic, courier, bryan, unwittingli, find, fig..."
1,1375441,Primetime,/lmrulvLbmaejTix1YaMxo1oGhH1.jpg,"[catch, predat, host, chri, hansen, set, make,..."
2,1283515,Verity,/rmaNMA1seHqpjtnYKzyynFAr1g8.jpg,"[lowen, ashleigh, hire, ghostwrit, best, sell,..."


In [31]:
df['tags']=df['tags'].apply(lambda x: " ".join(x))

In [32]:
df.reset_index(drop=True,inplace=True)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=3000,ngram_range=(1,2))
vectors=tfidf.fit_transform(df['tags'])

In [34]:
vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1162 stored elements and shape (20, 1016)>

In [35]:
tfidf.get_feature_names_out()

array(['abandon', 'abandon team', 'abduct', ..., 'young princ', 'zero',
       'zero godzilla'], shape=(1016,), dtype=object)

In [36]:
from sklearn.metrics.pairwise import cosine_similarity
similarities=cosine_similarity(vectors)

In [37]:
similarities[0]

array([1.        , 0.        , 0.        , 0.04141618, 0.        ,
       0.01804273, 0.01617412, 0.00773295, 0.00722005, 0.01963897,
       0.        , 0.03471415, 0.04612889, 0.02479508, 0.        ,
       0.0171247 , 0.00817709, 0.00771637, 0.00672046, 0.        ])

In [38]:
sorted(enumerate(similarities[0]),reverse=True,key=lambda x:x[1])[1:6]

[(12, np.float64(0.04612889114384075)),
 (3, np.float64(0.04141618093552173)),
 (11, np.float64(0.0347141530453923)),
 (13, np.float64(0.024795077351527937)),
 (9, np.float64(0.019638972055149116))]

In [39]:
df.head(1)

,id,title,poster_path,tags
0,1423191,Resident Evil,/gAgWJhUSQWCgtKR8170nMkYlBda.jpg,medic courier bryan unwittingli find fight sur...


In [40]:
df.to_csv('movies.csv')

In [48]:
list(df['title'].unique())

20

In [42]:
import joblib
with open('similarity.pkl','wb') as file:
  joblib.dump(similarities,file)

In [43]:
df[df['title']=='Resident Evil'].index[0]

np.int64(0)

In [44]:
df[df.index==435]['title']

,title
